In [43]:
import sys
!{sys.executable} -m pip install anthropic
import anthropic
import pandas as pd


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
print("Connected to Claude!!")



Connected to Claude!!


In [11]:

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What are the 3 most common data quality issues in fintech datasets?"}
    ]
)
print(response.content[0].text)

# 3 Most Common Data Quality Issues in Fintech Datasets

## 1. **Missing or Incomplete Transactions**
- Gaps in transaction records due to system failures, API downtime, or batch processing delays
- Incomplete customer profiles (missing KYC data, phone numbers, addresses)
- **Impact**: Breaks audit trails, complicates fraud detection, creates compliance risks

## 2. **Data Latency and Timeliness Issues**
- Real-time data lagging behind actual market conditions or transactions
- Batch processing delays causing stale information in risk models
- Settlement delays creating discrepancies between recorded vs. actual balances
- **Impact**: Poor decision-making, inaccurate risk assessments, regulatory reporting errors

## 3. **Inconsistent Data Formats and Duplicates**
- Different systems using varied standards (currencies, date formats, account identifiers)
- Duplicate customer records or transactions from multiple data sources
- Inconsistent transaction categorization across platforms
- **I

In [16]:
df= pd.read_csv('fintech_transactions_messy.csv')
nulls= df.isna().sum().to_dict()
dtypes= df.dtypes.astype(str).to_dict()
shape = df.shape
columns= list(df.columns)
print(nulls)
print(dtypes)
print(shape)
print(columns)

{'Transaction ID': 0, 'User ID': 0, 'Transaction Type': 0, ' Amount (USD)': 25, 'Merchant': 35, 'Currency': 35, 'Status': 36, 'Device': 35, 'Country': 35, 'Fee (USD)': 44, 'User Age': 22, 'Timestamp': 0, 'Is Flagged': 14}
{'Transaction ID': 'object', 'User ID': 'object', 'Transaction Type': 'object', ' Amount (USD)': 'float64', 'Merchant': 'object', 'Currency': 'object', 'Status': 'object', 'Device': 'object', 'Country': 'object', 'Fee (USD)': 'float64', 'User Age': 'float64', 'Timestamp': 'object', 'Is Flagged': 'float64'}
(355, 13)
['Transaction ID', 'User ID', 'Transaction Type', ' Amount (USD)', 'Merchant', 'Currency', 'Status', 'Device', 'Country', 'Fee (USD)', 'User Age', 'Timestamp', 'Is Flagged']


In [19]:
import json

nulls= df.isna().sum().to_dict()
dtypes= df.dtypes.astype(str).to_dict()
duplicates= int(df.duplicated().sum())
shape= df.shape
columns= list(df.columns)

samples= {col: df[col].dropna().head(3).tolist() for col in df.columns}

prompt = f"""
You are a data quality inspector. Analyze this dataset and return a full data quality report.

Dataset: {shape[0]} rows x {shape[1]} columns
Columns: {columns}
Duplicate rows: {duplicates}

Null counts per column:
{json.dumps(nulls, indent=2)}

Data types per column:
{json.dumps(dtypes, indent=2)}

Sample values per column:
{json.dumps(samples, indent=2)}

Return a detailed report covering:
1. Which columns have missing values and how bad it is
2. Which columns have inconsistent formatting or casing issues
3. Which columns have outliers or wrong values
4. Duplicate row situation
5. Top 5 cleaning steps to fix everything, in priority order
"""
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1500,
    messages=[{"role": "user", "content": prompt}]
)

print(response.content[0].text)


# Data Quality Report

## 1. Missing Values Analysis

| Column | Null Count | % Missing | Severity |
|--------|-----------|-----------|----------|
| Fee (USD) | 44 | 12.4% | **HIGH** |
| Status | 36 | 10.1% | **HIGH** |
| Merchant | 35 | 9.9% | **HIGH** |
| Currency | 35 | 9.9% | **HIGH** |
| Device | 35 | 9.9% | **HIGH** |
| Country | 35 | 9.9% | **HIGH** |
| Amount (USD) | 25 | 7.0% | **MEDIUM** |
| User Age | 22 | 6.2% | **MEDIUM** |
| Is Flagged | 14 | 3.9% | **LOW** |
| Transaction ID | 0 | 0% | ✓ |
| User ID | 0 | 0% | ✓ |
| Timestamp | 0 | 0% | ✓ |
| Transaction Type | 0 | 0% | ✓ |

**Summary:** 281 missing values total (10.6% of dataset). Six columns with ~10% missing data suggests possible data extraction or integration issues.

---

## 2. Formatting & Consistency Issues

| Column | Issue | Evidence | Severity |
|--------|-------|----------|----------|
| **Transaction Type** | Inconsistent whitespace | `"  Transfer  "` has leading/trailing spaces | **MEDIUM** |
| **Amount (USD

In [25]:
import json, re, base64, io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [26]:
nulls=df.isna().sum().to_dict()
null_pct=(df.isna().mean()*100).round(1).to_dict()
dtypes=df.dtypes.astype(str).to_dict()
dupes=int(df.duplicated().sum())
shape=df.shape
columns=list(df.columns)
samples={col: df[col].dropna().head(3).tolist() for col in df.columns}
num_cols=df.select_dtypes(include='number').columns.tolist()

In [27]:
format_issues={}
for col in df.select_dtypes(include='object').columns:
    issues=[]
    s=df[col].dropna().astype(str)
    if s.str.strip().ne(s).any(): issues.append('leading/trailing whitespace')
    if s.str.contains(r'[!@#$%^&*(){}\[\]<>]',regex=True).any(): issues.append('special characters')
    mixed=(s.str.islower().any() and s.str.isupper().any()) or s.str.istitle().ne(s.str.islower()).any()
    vals=s.str.lower().value_counts()
    if len(vals)>1:
        for v in vals.index:
            if v.strip().lower() in [x.strip().lower() for x in vals.index if x!=v]:
                issues.append('case inconsistency')
                break
    if issues: format_issues[col]=list(set(issues))

neg_issues={}
for col in num_cols:
    if (df[col].dropna()<0).any():
        n=int((df[col].dropna()<0).sum())
        neg_issues[col]=f'{n} negative values'

null_complete={'complete':int((df.notna().all(axis=1)).sum()),'has_nulls':int((df.isna().any(axis=1)).sum())}



In [36]:
prompt=f"""
Return only valid JSON.

Dataset:{shape[0]} rows x {shape[1]} columns
Columns:{columns}
Duplicates: {dupes}
Null counts: {json.dumps(nulls)}
Null %: {json.dumps(null_pct)}
Dtypes: {json.dumps(dtypes)}
Samples: {json.dumps(samples)}
Format issues detected: {json.dumps(format_issues)}
Negative value issues: {json.dumps(neg_issues)}

Return this JSON:
{{
  "dataset_description": "2-3 sentence description of what this dataset likely contains based on column names and sample values",
  "quality_score": 0,
  "overall_rating": "Good | Fair | Poor",
  "total_issues": 0,
  "columns": [
    {{
      "name": "col",
      "dtype": "dtype",
      "description": "one sentence what this column represents",
      "null_count": 0,
      "null_pct": 0.0,
      "issues": ["specific issue description"],
      "severity": "critical | moderate | minor | passed",
      "steps": [
        {{
          "text": "plain English action",
          "command": "df['col'].fillna(df['col'].median(), inplace=True)"
        }}
      ]
    }}
  ],
  "priority_actions": [
    {{
      "rank": 1,
      "action": "cleaning action title",
      "command": "df.drop_duplicates(inplace=True)",
      "columns_affected": ["col"]
    }}
  ]
}}

Rules:
- quality_score 0-100
- Include ALL columns
- issues=[] and steps=[] if severity ok
- steps: 1-3 items, each with plain text + real pandas command
- priority_actions: exactly 5, ranked by impact, each with a real pandas command
- dataset_description: infer from column names and sample values, be specific
- Flag format issues, negative values, dtype mismatches, whitespace, casing problems
"""

In [45]:
resp=client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=3000,
    messages=[{'role':'user','content':prompt}]
)
raw=re.sub(r'^```json\s*','',resp.content[0].text.strip())
raw=re.sub(r'```$','',raw).strip()
report=json.loads(raw)

In [32]:
BG='#08080F'
SURFACE='#0D0D18'
BORDER='#1A1A2E'
TEXT='#C8D0F0'
MUTED='#4A5080'

def fig_to_b64(fig):
    buf=io.BytesIO()
    fig.savefig(buf,format='png',dpi=160,bbox_inches='tight',facecolor=BG,edgecolor='none')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=MUTED,labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.set_axisbelow(True)

null_data={k:v for k,v in null_pct.items() if v>0}
fig1,ax1=plt.subplots(figsize=(7,max(2.8,len(null_data)*0.55)))
fig1.patch.set_facecolor(BG)
style_ax(ax1)
if null_data:
    cs=sorted(null_data,key=null_data.get,reverse=True)
    vs=[null_data[c] for c in cs]
    bc=['#7B4FD4' if v>15 else '#4A90D9' if v>8 else '#6B7FD4' for v in vs]
    bars=ax1.barh(cs,vs,color=bc,height=0.5,zorder=2)
    for bar,val in zip(bars,vs):
        ax1.text(bar.get_width()+0.2,bar.get_y()+bar.get_height()/2,
                 f'{val}%',va='center',color='#A0A8C8',fontsize=8)
    ax1.set_xlim(0,max(vs)*1.3)
    ax1.axvline(x=5,color=BORDER,linestyle='--',linewidth=0.8,zorder=1)
else:
    ax1.text(0.5,0.5,'No missing values',ha='center',va='center',
             color='#4CAF7D',fontsize=11,transform=ax1.transAxes)
ax1.set_xlabel('Missing (%)',color=MUTED,fontsize=8)
ax1.set_title('Missing values by column',color=TEXT,fontsize=10,fontweight='500',loc='left',pad=8)
ax1.xaxis.grid(True,color=BORDER,linewidth=0.6)
fig1.tight_layout()
chart1_b64=fig_to_b64(fig1)
plt.close(fig1)

sev_counts={'critical':0,'moderate':0,'minor':0,'passed':0}
for c in report['columns']:
    sv=c.get('severity','passed')
    sev_counts[sv]=sev_counts.get(sv,0)+1

sev_clrs={'critical':'#D4506A','moderate':'#7B4FD4','minor':'#4A90D9','passed':'#2A5FA8'}
pie_labels=[k for k,v in sev_counts.items() if v>0 and k!='passed']
label_names={'critical':'Critical','moderate':'Moderate','minor':'Minor','passed':'Passed'}
pie_vals=[sev_counts[k] for k in pie_labels]
pie_colors=[sev_clrs[k] for k in pie_labels]

legend_patches=[mpatches.Patch(color=sev_clrs[l],label=f"{label_names.get(l,l)}  {sev_counts[l]}") for l in pie_labels]

fig2,ax2=plt.subplots(figsize=(4.5,4))
fig2.patch.set_facecolor(BG)
ax2.set_facecolor(BG)
if pie_vals:
    wedges,texts,autotexts=ax2.pie(
        pie_vals,labels=None,colors=pie_colors,autopct='%1.0f%%',
        startangle=90,pctdistance=0.72,
        wedgeprops={'linewidth':2,'edgecolor':BG}
    )
    for at in autotexts:
        at.set_color(BG)
        at.set_fontsize(9)
        at.set_fontweight('600')
    legend_patches=[mpatches.Patch(color=sev_clrs[l],label=f"{label_names.get(l,l)}  {sev_counts[l]}") for l in pie_labels]
    ax2.legend(handles=legend_patches,loc='lower center',bbox_to_anchor=(0.5,-0.08),
               ncol=2,frameon=False,fontsize=8,labelcolor='#8090B0')
ax2.set_title('Issue severity distribution',color=TEXT,fontsize=10,fontweight='500',pad=10)
fig2.tight_layout()
chart2_b64=fig_to_b64(fig2)
plt.close(fig2)

chart_dist_b64=None
show_cols=num_cols[:4]
if show_cols:
    n=len(show_cols)
    fig3,axes=plt.subplots(1,n,figsize=(5*n,4.2))
    fig3.patch.set_facecolor(BG)
    if n==1: axes=[axes]
    pal=['#4A90D9','#7B4FD4','#6B7FD4','#9B6FE4']
    for i,(col,ax) in enumerate(zip(show_cols,axes)):
        style_ax(ax)
        data=df[col].dropna()
        q99=data.quantile(0.99)
        data=data[data<=q99]
        ax.hist(data,bins=25,color=pal[i%len(pal)],alpha=0.82,edgecolor=BG,linewidth=0.3)
        ax.set_title(col,color=TEXT,fontsize=10,fontweight='500')
        ax.set_ylabel('count',color=MUTED,fontsize=8)
        ax.xaxis.grid(True,color=BORDER,linewidth=0.4)
        mean_val=data.mean()
        ax.axvline(mean_val,color='#E8A838',linewidth=1.2,linestyle='--',alpha=0.8)
        ax.text(mean_val,ax.get_ylim()[1]*0.92,f' mean\n {mean_val:.1f}',
                color='#E8A838',fontsize=7,va='top')
    fig3.suptitle('Numeric column distributions',
                  color=MUTED,fontsize=8,y=1.01)
    fig3.tight_layout()
    chart_dist_b64=fig_to_b64(fig3)
    plt.close(fig3)

In [ ]:
score=report['quality_score']
rating=report['overall_rating']
dataset_desc=report.get('dataset_description','')
score_color='#FF6EB4' if score>=80 else '#FF9ECC' if score>=60 else '#FF4D8F'

sev_colors={'critical':'#FF4D8F','moderate':'#C966CC','minor':'#FF9ECC','passed':'#B06ECC'}
sev_bg={'critical':'rgba(255,77,143,0.13)','moderate':'rgba(201,102,204,0.13)',
        'minor':'rgba(255,158,204,0.13)','passed':'rgba(176,110,204,0.13)'}

rows_html=''
for c in report['columns']:
    sv=c.get('severity','passed')
    sev_label={'critical':'CRITICAL','moderate':'MODERATE','minor':'MINOR','passed':'PASSED'}.get(sv,'OK')
    issues_str='; '.join(c.get('issues',[])) if c.get('issues') else 'None detected'
    steps=c.get('steps',[])
    steps_html=''
    for s in steps:
        txt=s.get('text','') if isinstance(s,dict) else str(s)
        cmd=s.get('command','') if isinstance(s,dict) else ''
        cmd_html=f'<code class="cmd">{cmd}</code>' if cmd else ''
        steps_html+=f'<li><span class="step-txt">{txt}</span>{cmd_html}</li>'
    if not steps_html: steps_html='<li><span class="step-txt">No action needed</span></li>'
    rows_html+=f"""
    <tr class="drow" onclick="toggleRow(this)">
      <td class="cn-cell">
        <span class="cn">{c['name']}</span>
        <span class="cdesc">{c['description']}</span>
      </td>
      <td><code class="dtype">{c['dtype']}</code></td>
      <td class="mono">{c['null_count']} <span class="dim">({c['null_pct']}%)</span></td>
      <td><span class="pill" style="color:{sev_colors[sv]};b
      ackground:{sev_bg[sv]}">{sev_label}</span></td>
      <td class="issues-td">{issues_str}</td>
      <td class="arr-td"><span class="arr">›</span></td>
    </tr>
    <tr class="detail-row">
      <td colspan="6">
        <div class="detail-inner">
          <span class="detail-label">Steps to fix</span>
          <ul class="steps-list">{steps_html}</ul>
        </div>
      </td>
    </tr>"""

actions_html=''
for a in report['priority_actions']:
    affected=', '.join(a.get('columns_affected',[]))
    cmd=a.get('command','')
    cmd_html=f'<code class="act-cmd">{cmd}</code>' if cmd else ''
    actions_html+=f"""
    <div class="action-row">
      <div class="action-rank">{a['rank']}</div>
      <div class="action-body">
        <div class="action-text">{a['action']}</div>
        <div class="action-cols">{affected}</div>
        {cmd_html}
      </div>
    </div>"""

dist_section=f'<div class="sec-label" style="margin-top:2rem">Numeric distributions</div><div class="chart-box-full"><img src="data:image/png;base64,{chart_dist_b64}" style="width:100%;border-radius:4px"></div>' if chart_dist_b64 else ''

html=f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<title>Data Quality Report</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'Inter',sans-serif;background:#0A0008;color:#F0D6E8;font-size:15px;line-height:1.6}}

.header{{background:#120010;border-bottom:1px solid #2E0A28;padding:1.8rem 2.5rem;display:grid;grid-template-columns:1fr auto;gap:2rem;align-items:start}}
.header-eyebrow{{font-size:0.65rem;font-weight:500;text-transform:uppercase;letter-spacing:0.16em;color:#C070A0;margin-bottom:0.4rem}}
.header-title{{font-size:2rem;font-weight:600;color:#FFD6EC;letter-spacing:-0.02em;margin-bottom:0.6rem}}
.header-desc{{font-size:0.82rem;color:#C080A0;line-height:1.6;max-width:680px}}
.score-block{{text-align:right;padding-top:0.2rem}}
.score-label{{font-size:0.62rem;text-transform:uppercase;letter-spacing:0.14em;color:#C070A0;margin-bottom:4px}}
.score-num{{font-size:3.8rem;font-weight:600;color:{score_color};line-height:1;letter-spacing:-0.03em}}
.score-rating{{font-size:0.68rem;color:#C070A0;margin-top:5px;text-transform:uppercase;letter-spacing:0.1em}}

.summary-bar{{background:#120010;border-bottom:1px solid #2E0A28;padding:1rem 2.5rem;display:flex;gap:2.5rem;align-items:center;flex-wrap:wrap}}
.s-stat{{display:flex;flex-direction:column}}
.s-num{{font-size:1.6rem;font-weight:600;color:#FFD6EC}}
.s-label{{font-size:0.68rem;text-transform:uppercase;letter-spacing:0.1em;color:#C070A0;margin-top:2px}}
.s-div{{width:1px;height:34px;background:#2E0A28}}
.sev-row{{display:flex;gap:1.4rem;flex-wrap:wrap}}
.sev-item{{display:flex;align-items:center;gap:6px;font-size:0.78rem;color:#D090B0}}
.sev-dot{{width:7px;height:7px;border-radius:50%;flex-shrink:0}}

.main{{padding:2rem 2.5rem}}
.sec-label{{font-size:0.68rem;font-weight:500;text-transform:uppercase;letter-spacing:0.16em;color:#C070A0;margin-bottom:1rem;padding-bottom:0.5rem;border-bottom:1px solid #2E0A28}}

.charts-top{{display:grid;grid-template-columns:2fr 1fr;gap:1rem;margin-bottom:1rem}}
.chart-box{{background:#120010;border:1px solid #2E0A28;border-radius:8px;padding:1.2rem;overflow:hidden}}
.chart-box-full{{background:#120010;border:1px solid #2E0A28;border-radius:8px;padding:1.2rem;overflow:hidden;margin-bottom:2rem}}

.table-wrap{{background:#120010;border:1px solid #2E0A28;border-radius:8px;overflow:hidden;margin-bottom:2rem}}
table{{width:100%;border-collapse:collapse;table-layout:fixed;font-size:13px}}
thead{{background:#0A0008}}
th{{padding:0.65rem 1rem;text-align:left;font-size:0.6rem;font-weight:500;text-transform:uppercase;letter-spacing:0.14em;color:#C070A0;white-space:nowrap}}
th:nth-child(1){{width:26%}}th:nth-child(2){{width:9%}}th:nth-child(3){{width:11%}}th:nth-child(4){{width:10%}}th:nth-child(5){{width:38%}}th:nth-child(6){{width:6%}}

.drow{{border-bottom:1px solid #1A0518;cursor:pointer;transition:background 0.1s}}
.drow:hover,.drow.open{{background:#160012}}
td{{padding:0.75rem 1rem;vertical-align:top}}
.cn-cell{{display:flex;flex-direction:column;gap:3px}}
.cn{{font-weight:500;font-size:13.5px;color:#FFD6EC;font-family:'IBM Plex Mono',monospace}}
.cdesc{{font-size:12px;color:#C080A0;line-height:1.4}}
.dtype{{font-family:'IBM Plex Mono',monospace;font-size:10.5px;background:#1E0518;color:#FF9ECC;padding:2px 7px;border-radius:3px;border:none}}
.mono{{font-family:'IBM Plex Mono',monospace;font-size:12px;color:#D090B0}}
.dim{{color:#A06080;font-size:11px}}
.pill{{font-size:10px;font-weight:600;padding:3px 9px;border-radius:3px;letter-spacing:0.06em}}
.issues-td{{font-size:13px;color:#C080A0;line-height:1.5;word-break:break-word}}
.arr-td{{text-align:center;vertical-align:middle}}
.arr{{font-size:18px;color:#6A2050;display:inline-block;transition:transform 0.2s;line-height:1}}
.drow.open .arr{{transform:rotate(90deg);color:#FF6EB4}}

.detail-row{{display:none;background:#080006;border-bottom:1px solid #1A0518}}
.detail-row.open{{display:table-row}}
.detail-inner{{padding:0.8rem 1rem 1rem 2rem;display:flex;align-items:flex-start;gap:1.5rem}}
.detail-label{{font-size:0.58rem;text-transform:uppercase;letter-spacing:0.14em;color:#C070A0;white-space:nowrap;margin-top:3px;min-width:70px}}
.steps-list{{list-style:none;display:flex;flex-direction:column;gap:8px;flex:1}}
.steps-list li{{display:flex;flex-direction:column;gap:4px}}
.step-txt{{font-size:13px;color:#D090B0;padding-left:1.2rem;position:relative}}
.step-txt::before{{content:'→';position:absolute;left:0;color:#FF6EB4;font-size:11px}}
.cmd{{display:block;font-family:'IBM Plex Mono',monospace;font-size:12px;color:#FF9ECC;background:#1E0518;padding:5px 10px;border-radius:4px;border-left:2px solid #8B2060;margin-top:2px;word-break:break-all}}

.actions-wrap{{display:flex;flex-direction:column;gap:8px;margin-bottom:2rem}}
.action-row{{background:#120010;border:1px solid #2E0A28;border-radius:6px;padding:1rem 1.3rem;display:flex;gap:1rem;align-items:flex-start;transition:background 0.1s}}
.action-row:hover{{background:#160012}}
.action-rank{{background:#1E0518;border:1px solid #4A1040;color:#FF9ECC;width:28px;height:28px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:600;font-size:0.8rem;flex-shrink:0;margin-top:2px}}
.action-body{{flex:1;display:flex;flex-direction:column;gap:5px}}
.action-text{{font-size:14px;font-weight:500;color:#FFD6EC}}
.action-cols{{font-family:'IBM Plex Mono',monospace;font-size:11px;color:#FF6EB4}}
.act-cmd{{font-family:'IBM Plex Mono',monospace;font-size:12px;color:#FF9ECC;background:#1E0518;padding:4px 9px;border-radius:4px;border-left:2px solid #8B2060;word-break:break-all;display:block}}
</style>
</head><body>

<div class="header">
  <div>
    <div class="header-eyebrow">Data Quality Report</div>
    <div class="header-title">{shape[0]:,} rows &times; {shape[1]} columns</div>
    <div class="header-desc">{dataset_desc}</div>
  </div>
  <div class="score-block">
    <div class="score-label">Quality Score</div>
    <div class="score-num">{score}</div>
    <div class="score-rating">{rating}</div>
  </div>
</div>

<div class="summary-bar">
  <div class="s-stat"><div class="s-num">{dupes}</div><div class="s-label">Duplicate rows</div></div>
  <div class="s-div"></div>
  <div class="s-stat"><div class="s-num">{sum(nulls.values()):,}</div><div class="s-label">Missing values</div></div>
  <div class="s-div"></div>
  <div class="s-stat"><div class="s-num">{null_complete['complete']:,}</div><div class="s-label">Complete rows</div></div>
  <div class="s-div"></div>
  <div class="s-stat"><div class="s-num">{report['total_issues']}</div><div class="s-label">Issues detected</div></div>
  <div class="s-div"></div>
  <div class="sev-row">
    <div class="sev-item"><div class="sev-dot" style="background:#FF4D8F"></div>{sev_counts.get('critical',0)} critical</div>
    <div class="sev-item"><div class="sev-dot" style="background:#C966CC"></div>{sev_counts.get('moderate',0)} moderate</div>
    <div class="sev-item"><div class="sev-dot" style="background:#FF9ECC"></div>{sev_counts.get('minor',0)} minor</div>
    <div class="sev-item"><div class="sev-dot" style="background:#B06ECC"></div>{sev_counts.get('passed',0)} passed</div>
  </div>
</div>

<div class="main">
  <div class="sec-label">Visualizations</div>
  <div class="charts-top">
    <div class="chart-box"><img src="data:image/png;base64,{chart1_b64}" style="width:100%;border-radius:4px"></div>
    <div class="chart-box"><img src="data:image/png;base64,{chart2_b64}" style="width:100%;border-radius:4px"></div>
  </div>
  {dist_section}

  <div class="sec-label" style="margin-top:2rem">Column analysis</div>
  <div class="table-wrap">
    <table>
      <thead><tr>
        <th>Column</th><th>Type</th><th>Missing</th><th>Severity</th><th>Issues</th><th></th>
      </tr></thead>
      <tbody>{rows_html}</tbody>
    </table>
  </div>

  <div class="sec-label">Priority cleaning actions</div>
  <div class="actions-wrap">{actions_html}</div>
</div>

<script>
function toggleRow(row){{
  const isOpen=row.classList.contains('open');
  document.querySelectorAll('.drow.open').forEach(r=>{{
    r.classList.remove('open');
    r.nextElementSibling.classList.remove('open');
  }});
  if(!isOpen){{
    row.classList.add('open');
    row.nextElementSibling.classList.add('open');
  }}
}}
</script>
</body></html>"""

with open('data_quality_dashboard.html','w') as f:
    f.write(html)
print('Done!')

Done!
